# 05 - Entrenamiento del Modelo Cognitivo Artificial

**Modelo Cognitivo Artificial para la Replicación de la Actividad Neurofisiológica de Percepción de Profundidad**

- **Autor:** Jesús Goenaga Peña
- **Programa:** Doctorado en Ciencias Cognitivas - Universidad Autónoma de Manizales

---

Entrenamiento en dos etapas según la sección 8.4.1.3 de la tesis:

| Etapa | Fases congeladas | Learning Rate | Objetivo |
|-------|-----------------|---------------|----------|
| 1 | NGL + V1 (7-15) | 0.001 | Optimizar capas superiores y salida |
| 2 | Ninguna | 0.0001 | Fine-tuning global del modelo completo |

## 1. Configuración

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess
import torch
import numpy as np
import matplotlib.pyplot as plt
import random

PROJECT_ROOT = '/content/drive/MyDrive/cognitive-depth-model'
REPO_DIR = '/content/cognitive-depth-model'

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/yisusgoenaga/cognitive-depth-model.git {REPO_DIR}

# Encontrar src automáticamente
result = subprocess.run(['find', REPO_DIR, '-name', 'cognitive_model.py'], capture_output=True, text=True)
model_path = result.stdout.strip()
src_dir = os.path.dirname(os.path.dirname(model_path))
sys.path.insert(0, src_dir)

# Reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')
print(f'Src: {src_dir}')

## 2. Cargar Datos

In [ ]:
from training.dataset import KITTIStereoDepthDataset, create_dataloaders

KITTI_RAW = os.path.join(PROJECT_ROOT, 'data/raw/kitti')
SPLITS_FILE = os.path.join(PROJECT_ROOT, 'data/splits/kitti_splits.json')

# Verificar que existen los archivos
print(f'KITTI path: {KITTI_RAW} -> Existe: {os.path.exists(KITTI_RAW)}')
print(f'Splits file: {SPLITS_FILE} -> Existe: {os.path.exists(SPLITS_FILE)}')

# Crear DataLoaders
dataloaders = create_dataloaders(
    kitti_base_path=KITTI_RAW,
    split_file=SPLITS_FILE,
    batch_size=8,          # Batch pequeño para GPU T4
    target_size=(256, 512),
    num_workers=2,
    seed=SEED
)

train_loader = dataloaders['train']
test_loader = dataloaders['test']

print(f'\nBatches de entrenamiento: {len(train_loader)}')
print(f'Batches de prueba: {len(test_loader)}')

In [ ]:
# Verificar un batch
sample_batch, sample_labels = next(iter(train_loader))

print(f'Batch de entrada:  {sample_batch.shape}  (batch, 6ch, H, W)')
print(f'Batch de labels:   {sample_labels.shape}')
print(f'Rango de entrada:  [{sample_batch.min():.3f}, {sample_batch.max():.3f}]')
print(f'Labels del batch:  {sample_labels.squeeze().tolist()}')

# Visualizar un ejemplo del batch
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Ejemplo del Batch de Entrenamiento', fontsize=14, fontweight='bold')

# Ojo izquierdo (canales 0-2)
left = sample_batch[0, :3].permute(1, 2, 0).numpy()
axes[0].imshow(left[:, :, ::-1])  # BGR a RGB
axes[0].set_title(f'Ojo Izquierdo\nLabel: {sample_labels[0].item():.1f}')
axes[0].axis('off')

# Ojo derecho (canales 3-5)
right = sample_batch[0, 3:].permute(1, 2, 0).numpy()
axes[1].imshow(right[:, :, ::-1])
axes[1].set_title('Ojo Derecho')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 3. Crear el Modelo

In [ ]:
from model.cognitive_model import create_model

model = create_model()
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Modelo creado: {total_params:,} parámetros')
print(f'Dispositivo: {device}')

# Test rápido
with torch.no_grad():
    test_output = model(sample_batch.to(device))
    print(f'Test forward pass: {test_output.shape} -> OK')

## 4. Entrenamiento

**Etapa 1:** Congelamos NGL y V1 (Fases 7-15). Solo entrenamos V2, V3, V4, V5/MT y la salida.
Esto permite que la capa de decisión aprenda primero sin alterar las representaciones tempranas.

**Etapa 2:** Descongelamos todo y hacemos fine-tuning con learning rate reducido.
Esto ajusta la coherencia global entre todas las fases.

In [ ]:
from training.trainer import train_model

CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, 'checkpoints')

print('Iniciando entrenamiento del Modelo Cognitivo Artificial')
print(f'Checkpoints se guardarán en: {CHECKPOINT_DIR}')
print()

history = train_model(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    device=device,
    checkpoint_dir=CHECKPOINT_DIR,
    stage1_epochs=50,     # Épocas con capas congeladas
    stage2_epochs=100,    # Épocas de fine-tuning
    stage1_lr=0.001,      # LR para Etapa 1
    stage2_lr=0.0001,     # LR reducido para Etapa 2
    patience=20,          # Early stopping
)

## 5. Visualización del Entrenamiento

In [ ]:
# Extraer métricas del historial
epochs = history['epoch']
stages = history['stage']
train_losses = [m['loss'] for m in history['train']]
test_losses = [m['loss'] for m in history['test']]
train_accs = [m['accuracy'] for m in history['train']]
test_accs = [m['accuracy'] for m in history['test']]
train_f1s = [m['f1'] for m in history['train']]
lrs = history['lr']

# Encontrar límite entre etapas
stage1_end = max(i for i, s in enumerate(stages) if s == 1) + 1 if 1 in stages else 0

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Historial de Entrenamiento del Modelo Cognitivo Artificial',
             fontsize=14, fontweight='bold')

# Loss
axes[0, 0].plot(epochs, train_losses, 'b-', label='Train', alpha=0.7)
axes[0, 0].plot(epochs, test_losses, 'r-', label='Test', alpha=0.7)
if stage1_end > 0:
    axes[0, 0].axvline(x=epochs[stage1_end-1], color='gray', linestyle='--', label='Etapa 1 → 2')
axes[0, 0].set_xlabel('Época')
axes[0, 0].set_ylabel('Loss (BCE)')
axes[0, 0].set_title('Función de Pérdida')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Accuracy
axes[0, 1].plot(epochs, train_accs, 'b-', label='Train', alpha=0.7)
axes[0, 1].plot(epochs, test_accs, 'r-', label='Test', alpha=0.7)
if stage1_end > 0:
    axes[0, 1].axvline(x=epochs[stage1_end-1], color='gray', linestyle='--', label='Etapa 1 → 2')
axes[0, 1].set_xlabel('Época')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].set_title('Precisión')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# F1 Score
axes[1, 0].plot(epochs, train_f1s, 'g-', label='Train F1', alpha=0.7)
if stage1_end > 0:
    axes[1, 0].axvline(x=epochs[stage1_end-1], color='gray', linestyle='--', label='Etapa 1 → 2')
axes[1, 0].set_xlabel('Época')
axes[1, 0].set_ylabel('F1 Score')
axes[1, 0].set_title('F1 Score (Train)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Learning Rate
axes[1, 1].plot(epochs, lrs, 'purple', alpha=0.7)
if stage1_end > 0:
    axes[1, 1].axvline(x=epochs[stage1_end-1], color='gray', linestyle='--', label='Etapa 1 → 2')
axes[1, 1].set_xlabel('Época')
axes[1, 1].set_ylabel('Learning Rate')
axes[1, 1].set_title('Learning Rate Schedule')
axes[1, 1].set_yscale('log')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'results/visualizations/training_history.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Historial guardado en results/visualizations/training_history.png')

## 6. Evaluación del Mejor Modelo

In [ ]:
from training.trainer import evaluate
import torch.nn as nn

# Cargar el mejor modelo
best_checkpoint = torch.load(os.path.join(CHECKPOINT_DIR, 'best_model.pth'))
model.load_state_dict(best_checkpoint['model_state_dict'])

print(f'Mejor modelo cargado (época {best_checkpoint["epoch"]}, etapa {best_checkpoint["stage"]})')
print(f'Test loss en ese punto: {best_checkpoint["test_loss"]:.4f}')

# Evaluar
criterion = nn.BCELoss()
test_metrics = evaluate(model, test_loader, criterion, device)

print(f'\nMétricas del mejor modelo en el conjunto de prueba:')
print(f'  Loss:      {test_metrics["loss"]:.4f}')
print(f'  Accuracy:  {test_metrics["accuracy"]:.4f}')
print(f'  Precision: {test_metrics["precision"]:.4f}')
print(f'  Recall:    {test_metrics["recall"]:.4f}')
print(f'  F1 Score:  {test_metrics["f1"]:.4f}')
print(f'  AUC:       {test_metrics["auc"]:.4f}')

In [ ]:
# Guardar métricas en CSV
import pandas as pd

# Historial completo
history_df = pd.DataFrame({
    'epoch': history['epoch'],
    'stage': history['stage'],
    'lr': history['lr'],
    'train_loss': [m['loss'] for m in history['train']],
    'train_accuracy': [m['accuracy'] for m in history['train']],
    'train_f1': [m['f1'] for m in history['train']],
    'test_loss': [m['loss'] for m in history['test']],
    'test_accuracy': [m['accuracy'] for m in history['test']],
    'test_f1': [m.get('f1', 0) for m in history['test']],
})

history_csv = os.path.join(PROJECT_ROOT, 'results/metrics/training_history.csv')
history_df.to_csv(history_csv, index=False)
print(f'Historial guardado en: {history_csv}')
print(f'\nÚltimas 5 épocas:')
print(history_df.tail())

In [ ]:
print('=' * 60)
print('NOTEBOOK 05 COMPLETADO')
print('=' * 60)
print()
print('Entrenamiento realizado:')
print(f'  Etapa 1: {stage1_end} épocas (NGL + V1 congelados)')
print(f'  Etapa 2: {len(epochs) - stage1_end} épocas (fine-tuning completo)')
print(f'  Total:   {len(epochs)} épocas')
print()
print('Archivos generados:')
print(f'  {os.path.join(CHECKPOINT_DIR, "best_model.pth")}')
print(f'  {os.path.join(CHECKPOINT_DIR, "model_final.pth")}')
print(f'  {history_csv}')
print(f'  results/visualizations/training_history.png')
print()
print('Siguiente: Notebook 06 - Evaluación y Explicabilidad')
print('  Grad-CAM, curvas ROC, análisis de activaciones con datos reales')